# Week 01 — Hugging Face basics

Mục tiêu: tokenization, inference và khảo sát file mô hình. Chọn model trong `configs/week01.json` trước khi chạy. Notebook này chưa được thực thi với model.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
config_path = ROOT / "configs" / "week01.json"
if not config_path.is_file():
    raise FileNotFoundError("Open Jupyter from the project root or notebooks directory")
config = json.loads(config_path.read_text(encoding="utf-8"))
if not config["model_id"]:
    raise ValueError("Set model_id in configs/week01.json before continuing")
config

## 1. Môi trường
Ghi lại phiên bản thư viện, thiết bị và seed vào nhật ký.

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:05:56) [MSC v.1942 64 bit (AMD64)]
d:\IDEs\Miniconda3\python.exe


In [ ]:
import importlib.metadata
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

for name in ["torch", "transformers", "datasets", "tokenizers", "safetensors"]:
    print(name, importlib.metadata.version(name))
print("CUDA:", torch.cuda.is_available())
set_seed(config["seed"])
device = config["device"]
if device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; use cpu or configure CUDA-enabled PyTorch")

## 2. Tokenizer
Lần đầu cần Internet. Quan sát token IDs, attention mask và kết quả giải mã.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config["model_id"])
encoded = tokenizer(config["prompt"], return_tensors="pt")
print(encoded)
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0].tolist()))
print(tokenizer.decode(encoded["input_ids"][0]))

## 3. Model và inference
Dùng greedy decoding để dễ so sánh. Khung này dành cho causal language model; mô hình chat có thể cần chat template riêng.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(config["model_id"])
model = model.to(device).eval()
inputs = {key: value.to(device) for key, value in encoded.items()}
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=config["max_new_tokens"],
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
    )
text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(text)

## 4. Khảo sát file trên Hugging Face
Lệnh dưới liệt kê tên file, không tải toàn bộ trọng số. Ghi nhận revision và các file thực tế vào `docs/model_files.md`.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
info = api.model_info(config["model_id"])
print("Repository revision:", info.sha)
print("Loaded model revision:", getattr(model.config, "_commit_hash", None))
for filename in api.list_repo_files(config["model_id"], revision=info.sha):
    print(filename)

## 5. Lưu kết quả
Mỗi lần chạy tạo một file riêng. Không lưu token truy cập vào kết quả.

In [ ]:
from datetime import datetime, timezone

output_dir = ROOT / "outputs" / "week01"
output_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
result = {
    "config": config, "generated_text": text,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "dtype": str(next(model.parameters()).dtype),
    "versions": {name: importlib.metadata.version(name) for name in ["torch", "transformers", "tokenizers"]},
}
output_path = output_dir / f"inference_{run_id}.json"
output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(output_path)

## 6. Ghi chú sau thực hành

- Thử thêm hai prompt và lưu kết quả từng lần.
- Giải thích vai trò của `transformers`, `tokenizers`, `datasets`.
- Phân biệt config, tokenizer, trọng số và model card.
- Ghi nhận dung lượng model, nơi cache và những lỗi đã xử lý.
- Hoàn thiện `docs/model_files.md` và `docs/experiment_log.md`.

**Nhận xét:** TODO